# Question 7: diabetes mellitus prediction with classes

In [10]:
from sklearn.metrics import roc_auc_score

import library_Group_M_M_J as mmj_lib

data_path = "data/sample_diabetes_mellitus_data.csv"
target = "diabetes_mellitus"

## Load

`CSVReader` reads the CSV and returns the train and test sets.

In [11]:
df_train, df_test = mmj_lib.CSVReader(data_path).read()
print(f"Train: {df_train.shape}, test: {df_test.shape}")

Train: (8000, 53), test: (2000, 53)


## Preprocess

Each preprocessor learns on train (`fit_transform`) and is only applied to test (`transform`),
so the means used to fill height and weight come from train.

In [12]:
preprocessors = [
    mmj_lib.NaNRowsRemover(["age", "gender", "ethnicity"]),
    mmj_lib.MeanImputer(["height", "weight"]),
]
for preprocessor in preprocessors:
    df_train = preprocessor.fit_transform(df_train)
    df_test = preprocessor.transform(df_test)

print(f"Train: {df_train.shape}, test: {df_test.shape}")

Train: (7494, 53), test: (1874, 53)


## Create the features

All features share the `Feature` interface

In [13]:
features_steps = [
    mmj_lib.DummiesFeature("ethnicity", drop_category="Caucasian"),
    mmj_lib.BinaryFeature("gender", {"M": 1, "F": 0}),
]
for feature in features_steps:
    df_train = feature.transform(df_train)
    df_test = feature.transform(df_test)

features = [
    "age", "height", "weight", "aids", "cirrhosis", "hepatic_failure",
    "immunosuppression", "leukemia", "lymphoma", "solid_tumor_with_metastasis",
    "gender_bin",
] + [col for col in df_train.columns if col.startswith("ethnicity_")]
features

['age',
 'height',
 'weight',
 'aids',
 'cirrhosis',
 'hepatic_failure',
 'immunosuppression',
 'leukemia',
 'lymphoma',
 'solid_tumor_with_metastasis',
 'gender_bin',
 'ethnicity_African American',
 'ethnicity_Asian',
 'ethnicity_Hispanic',
 'ethnicity_Native American',
 'ethnicity_Other/Unknown']

## Train and predict

In [14]:
model = mmj_lib.LogisticRegressionModel(features, target, hyperparameters={"C": 1.0})
model.train(df_train)

df_train["predictions"] = model.predict(df_train)
df_test["predictions"] = model.predict(df_test)
df_test[[target, "predictions"]].head()

,diabetes_mellitus,predictions
6252,1,0.309105
1731,0,0.264794
4742,0,0.099522
4521,0,0.162204
6340,1,0.487163


## Evaluate

In [15]:
print(f"Train ROC AUC: {roc_auc_score(df_train[target], df_train['predictions']):.3f}")
print(f"Test ROC AUC: {roc_auc_score(df_test[target], df_test['predictions']):.3f}")

Train ROC AUC: 0.692
Test ROC AUC: 0.668
